# GLS Lesion Segmentation — Colab Training Runner

Before running anything: **Runtime → Change runtime type → T4 GPU** (or better).

Run the cells below in order, top to bottom.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/p4ntomath/gls-lesion-segmentation.git
%cd gls-lesion-segmentation

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt -q

Confirm you actually have a GPU before continuing — training on CPU will be very slow.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. (Recommended) Mount Google Drive for persistence

Colab sessions are ephemeral — if you disconnect mid-training, anything not on Drive is lost,
including the downloaded/preprocessed dataset. Run this now, before preprocessing, so
`data/processed`, `data/splits`, and `outputs` all live on Drive instead of the throwaway VM disk.

Skip this cell only if you're doing a short one-off test run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/gls-data/processed
!mkdir -p /content/drive/MyDrive/gls-data/splits
!mkdir -p /content/drive/MyDrive/gls-data/outputs

!rm -rf data/processed data/splits outputs
!ln -s /content/drive/MyDrive/gls-data/processed data/processed
!ln -s /content/drive/MyDrive/gls-data/splits data/splits
!ln -s /content/drive/MyDrive/gls-data/outputs outputs

## 4. Upload `dataset.json`

`.gitignore` correctly excludes this from the repo, so it's not in the clone.
Run this cell, then pick `GLS_instanceSegmentationEasy-FinalRelease.json` from your machine
in the upload dialog that appears.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs("data/raw/annotations", exist_ok=True)
uploaded = files.upload()
uploaded_name = list(uploaded.keys())[0]
shutil.move(uploaded_name, "data/raw/annotations/dataset.json")
print("Saved to data/raw/annotations/dataset.json")

## 5. Run preprocessing

Parses the JSON, downloads every image + mask from Segments.ai's S3, decodes the binary
lesion masks, and writes the stratified train/val/test split. With 438 labeled samples this
will take a few minutes — it's downloading real files over the network.

In [ ]:
!python scripts/preprocess.py --config configs/base.yaml

Quick sanity check on what got written before you commit to a full training run.

In [ ]:
import pandas as pd

manifest = pd.read_csv("data/processed/manifest.csv")
print(manifest.shape)
print(manifest[["sample_id", "lesion_coverage"]].describe())

for split in ["train", "val", "test"]:
    with open(f"data/splits/{split}.txt") as f:
        print(split, "samples:", len(f.read().strip().splitlines()))

## 6. Train Experiment 1 (U-Net, no augmentation)

This is the first of the four experiments in your proposal's Table 1.

In [ ]:
!python scripts/train.py --experiment exp01_unet_noaug

Check what got saved.

In [ ]:
import pandas as pd

log = pd.read_csv("outputs/logs/exp01_unet_noaug.csv")
print(log.tail())

import os
ckpt_path = "outputs/checkpoints/exp01_unet_noaug.pt"
print(f"\ncheckpoint size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB")

## 7. (If you skipped Drive in step 3) Download your results manually

Only needed if you didn't mount Drive — otherwise your checkpoint and log are already
persisted there.

In [ ]:
from google.colab import files

files.download("outputs/checkpoints/exp01_unet_noaug.pt")
files.download("outputs/logs/exp01_unet_noaug.csv")

## 8. Run the remaining experiments

Once Experiment 1 looks right, run the other three the same way:

In [ ]:
!python scripts/train.py --experiment exp02_unet_aug
!python scripts/train.py --experiment exp03_attnunet_noaug
!python scripts/train.py --experiment exp04_attnunet_aug

---
## 9. Visualize results

Pull the latest code first if you've pushed `viz.py` / the fixed `evaluate.py`
to GitHub since you cloned:

In [ ]:
!git pull

### 9a. Training curves

Loss + validation Dice/IoU over epochs, straight from the training log.

In [ ]:
import matplotlib.pyplot as plt
from src.utils.viz import plot_training_curves

fig = plot_training_curves(
    "outputs/logs/exp01_unet_noaug.csv",
    save_path="outputs/figures/exp01_curves.png",
)
plt.show()

### 9b. Coverage scatter

Predicted vs. reference leaf-area GLS coverage, one point per test sample.
Needs `results.json` from `scripts/evaluate.py` — run that first if you
haven't (see earlier cell).

In [ ]:
import json
from src.utils.viz import plot_coverage_scatter

with open("experiments/exp01_unet_noaug/results.json") as f:
    results = json.load(f)

print("summary:", results["summary"])

fig = plot_coverage_scatter(results["per_image"], save_path="outputs/figures/exp01_coverage.png")
plt.show()

### 9c. Prediction grid (qualitative overlays)

This one needs actual images + masks + a fresh model inference pass, not just
`results.json`'s numbers. The cell below loads a handful of test samples,
runs the trained checkpoint, and shows Image / Ground Truth / Prediction /
Error Map side by side for each.

By default it shows the **worst 6 samples by Dice** — the most useful ones to
actually look at, since the good ones don't tell you much about failure
modes.

In [ ]:
import json
from pathlib import Path

import albumentations as A
import numpy as np
import torch

from src.data.augmentations import get_eval_transforms
from src.data.dataset import GLSDataset
from src.evaluation.evaluate import load_experiment_config, build_model
from src.utils.viz import plot_prediction_grid

EXPERIMENT = "exp01_unet_noaug"
N_SHOW = 6

config = load_experiment_config(EXPERIMENT, config_path="configs/base.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = config["data"]["image_size"]
threshold = config["training"].get("threshold", 0.5)

model = build_model(config)
ckpt = torch.load(f"outputs/checkpoints/{EXPERIMENT}.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

with open(f"experiments/{EXPERIMENT}/results.json") as f:
    results = json.load(f)

# worst N by Dice -- most informative samples to actually inspect
worst = sorted(results["per_image"], key=lambda r: r["dice"])[:N_SHOW]
sample_ids = [r["sample_id"] for r in worst]
titles = [f"{r['sample_id']}  dice={r['dice']:.3f}" for r in worst]

# Two transforms on the SAME raw image: one normalized (what the model
# actually sees, identical to evaluate.py's get_eval_transforms), one
# resize-only (for display -- imshow needs real uint8 pixel values, not
# ImageNet-normalized ones).
model_transform = get_eval_transforms(image_size)
display_resize = A.Compose([A.Resize(image_size, image_size)])

images_dir = Path(config["paths"]["processed_images_dir"])
masks_dir = Path(config["paths"]["lesion_masks_dir"])

images, gt_masks, pred_masks = [], [], []
for sid in sample_ids:
    raw_image = GLSDataset._load_rgb(images_dir / f"{sid}.jpg")
    raw_mask = GLSDataset._load_mask(masks_dir / f"{sid}.png")

    model_input = model_transform(image=raw_image, mask=raw_mask)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(model_input)
        pred = (torch.sigmoid(logits) >= threshold).cpu().numpy()[0, 0].astype(np.uint8)

    display = display_resize(image=raw_image, mask=raw_mask)
    images.append(display["image"])
    gt_masks.append(display["mask"])
    pred_masks.append(pred)

fig = plot_prediction_grid(images, gt_masks, pred_masks, n=N_SHOW, titles=titles,
                            save_path=f"outputs/figures/{EXPERIMENT}_worst_predictions.png")
plt.show()